# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library, focusing on a step-by-step data science workflow.

### Dataset Source

The dataset is provided in [Croissant](https://mlcommons.org/croissant/) format:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed in this environment
!pip install -U mlcroissant

## 1. Data Loading

Load dataset metadata and explore basic description and title.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# URL to the Croissant schema
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load Croissant Dataset (schema and data descriptors)
dataset = mlc.Dataset(croissant_url)
print(f"Dataset Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")
# Optional: show available license and version
print(f"License: {dataset.metadata.license}")
print(f"Version: {dataset.metadata.version}")

## 2. Data Overview

Review available record sets, their `@id`s, and the fields/columns they provide. All entities are referenced by their `@id` as per Croissant best practices.

In [ ]:
# List all available Record Sets in the dataset
record_sets = dataset.metadata.record_sets
print("Available Record Sets:")
for rs in record_sets:
    print(f"  - name: {rs.name}; @id: {rs['@id']}")

# For illustration, let's inspect fields and columns from the primary RecordSet, using @id reference.
if record_sets:
    main_rs = record_sets[0]
    print(f"\nSelected RecordSet: {main_rs.name} (@id: {main_rs['@id']})")
    print("Fields (by @id):")
    for field in main_rs.fields:
        print(f"  - {field.name}: {field['@id']} | DataType: {getattr(field, 'data_type', 'N/A')}")
    # Show columns and their IDs if present
    if hasattr(main_rs, 'columns') and main_rs.columns:
        print("Columns:")
        for col in main_rs.columns:
            print(f"  - {col.name}: {col['@id']}")

## 3. Data Extraction

Extract records from record sets using their `@id` and load them into pandas DataFrames for analysis.

In [ ]:
dataframes = {}
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
print("Loading each record set and their first 3 rows...")

for rs_id in record_set_ids:
    # The record_set parameter must be the @id
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"\nRecordSet @id: {rs_id}")
    print("Columns:", df.columns.tolist())
    print(df.head(3))

# For demonstration, select the first RecordSet found for further analysis (replace with domain-specific @id if needed)
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print(f"\nMain RecordSet selected for EDA: {main_record_set_id}")

## 4. Exploratory Data Analysis (EDA)

Apply filtering, normalization, and simple aggregations on a numeric field, referencing fields by their `@id`.

For demonstration, we'll:
- Select a numeric field from the first record set (for example, 'peptide_count' or 'mw' if present).
- Filter for entries with high values.
- Normalize, and group by a categorical field if available.

In [ ]:
# Identify a numeric field @id and a group field from the main record set's fields
main_rs = dataset.metadata.record_sets[0]
fields = {field['@id']: field for field in main_rs.fields}

df = dataframes[main_record_set_id]

# Heuristic to pick a representative numeric field (e.g., molecular weight or peptide count)
import numpy as np
numeric_field_id = None
group_field_id = None
# Try common names seen in proteomics datasets
for field in main_rs.fields:
    if field.data_type and field.data_type in ['schema:Float', 'schema:Number', 'schema:Integer']:
        # Use the first numeric field found
        if numeric_field_id is None:
            numeric_field_id = field['@id']
    if not group_field_id and (field.data_type and 'Text' in field.data_type):
        if 'acc' in field.name.lower() or 'gene' in field.name.lower() or 'protein' in field.name.lower():
            group_field_id = field['@id']

# Use DataFrame columns matching the @id
if numeric_field_id and numeric_field_id in df.columns:
    print(f"Using numeric field: {numeric_field_id}")
else:
    # Fallback: pick first numeric column
    for col in df.columns:
        if np.issubdtype(df[col].dtype, np.number):
            numeric_field_id = col
            print(f"Using fallback numeric field: {numeric_field_id}")
            break

filtered_df = df[df[numeric_field_id] > df[numeric_field_id].quantile(0.9)]  # Top 10% as threshold for demo
print(f"Filtered records with {numeric_field_id} in top 10%:")
print(filtered_df.head())

# Normalize the numeric field (Z-score)
filtered_df = filtered_df.copy()
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())


if group_field_id and group_field_id in df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame(name='mean_' + numeric_field_id)
    print(f"\nGrouped data by {group_field_id} (mean of {numeric_field_id}):")
    print(grouped_df.head())

## 5. Visualization

Visualize numeric field distributions and relationships using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,6))
        sns.boxplot(data=filtered_df, x=group_field_id, y=numeric_field_id)
        plt.title(f'{numeric_field_id} by {group_field_id} (Top 10% Records)')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion

In this notebook, we've demonstrated how to load, inspect, and analyze the FAIR^2 mass spectrometry dataset using the `mlcroissant` library. By referring to entities via `@id` throughout, this workflow ensures compatibility with Croissant and reusability of code for further semantic analyses. You can easily extend these steps to more advanced bioinformatic or proteomics pipelines using the same framework.